In [348]:
from pathlib import Path
 
import numpy as np
import pandas as pd

In [372]:
STRINGEE_DIR = Path(r"C:\Users\Administrator\Videos\Office\Dialer\CDR\Stringee")
UNO_DIR = Path(r"C:\Users\Administrator\Videos\Office\Dialer\CDR\Uno")        # check where your Uno export is
PROCESSED_DIR = Path(r"C:\Users\Administrator\Videos\Office\Dialer\Processed CDR") # cleaned copies of the raw file
REPORT_DIR = Path(r"C:\Users\Administrator\Videos\Office\Dialer\Report")
TEAM_LIST = Path(r"C:\Users\Administrator\Videos\Office\Dialer\Team List\20261030_133300.csv")
SAVE_PROCESSED_RAW = True  

In [373]:
def read_xlsx(path: Path) -> pd.DataFrame:
    """Read with the fast calamine engine if installed, otherwise pandas' default."""
    try:
        return pd.read_excel(path, engine="calamine")
    except ImportError:
        return pd.read_excel(path)

def write_xlsx(df: pd.DataFrame, path: Path) -> None:
    """Write with xlsxwriter (faster) if installed, otherwise pandas' default."""
    try:
        df.to_excel(path, index=False, engine="xlsxwriter")
    except ImportError:
        df.to_excel(path, index=False)

In [374]:
EXCLUDE_STRINGEE_END_CODES = []   # your first version excluded these; [] keeps them
DEDUPE_NUMBER_AND_TIME = True      # your first version dropped repeated Number + Call Start Time
UNO_CONNECTED_RULE = "answered_flag"   # "answered_flag" = Answered is TRUE (current); "both_answered" = Status

In [375]:
WORK_START = pd.Timedelta("09:30:00")    # call gaps are only flagged between these times
WORK_END = pd.Timedelta("18:30:00")
GAP_FLAG_SECONDS = 60                    # a gap longer than this counts as a 'Call Gap'
BREAK_ALLOWANCE = 3600                   # 1 hour taken off the total gap (when the gap is over 1 hour) and add

In [376]:
LEVERAGE_PER_CALL = 45                   # seconds of gap allowed per dialed call
LONG_CALL_SECONDS = 30                   # threshold for the _GT_30 columns
ATTENDANCE_BANDS = [(pd.Timedelta(hours=4, minutes=30), "Absent"),
                    (pd.Timedelta(hours=6), "Half Day"),
                    (pd.Timedelta(hours=8, minutes=30), "Warning")]   # at or above the last band: Present
KEEP_ATTENDANCE = False 

In [377]:
LEVERAGE_PER_CALL = 45                   # seconds of gap allowed per dialed call
LONG_CALL_SECONDS = 30                   # threshold for the _GT_30 columns
ATTENDANCE_BANDS = [(pd.Timedelta(hours=4, minutes=30), "Absent"),
                    (pd.Timedelta(hours=6), "Half Day"),
                    (pd.Timedelta(hours=8, minutes=30), "Warning")]   # at or above the last band: Present
KEEP_ATTENDANCE = False   

In [378]:
SLOT_TARGET = 25                         # calls or talk minutes in an hour for 'Complete'
POOL1_WARNING = (22, 24)                 # Pool 1: 'Warning' range (inclusive)
OTHER_WARNING = (22, 22)                 # other pools: 'Warning' range (inclusive)
SLOT_LABELS = ["0 to 8 AM", "8 to 9 AM", "9 to 10 AM", "10 to 11 AM", "11 to 12 PM", "12 to 13 PM",
               "13 to 14 PM", "14 to 15 PM", "15 to 16 PM", "16 to 17 PM", "17 to 18 PM",
               "18 to 19 PM", "19 to 20 PM", "20+"]


## **Helpers** ##

In [379]:
def read_xlsx(path, **kwargs):
    """Read with the fast calamine engine if installed, otherwise pandas' default."""
    try:
        return pd.read_excel(path, engine="calamine", **kwargs)
    except ImportError:
        return pd.read_excel(path, **kwargs)
 
 
def write_xlsx(df, path):
    """Write with xlsxwriter (faster) if installed, otherwise pandas' default."""
    try:
        df.to_excel(path, index=False, engine="xlsxwriter")
    except ImportError:
        df.to_excel(path, index=False)
 
 
def date_range_name(prefix, dates, ext):
    first, last = dates.min(), dates.max()
    if first.date() == last.date():
        return f"{prefix}_{first:%Y-%m-%d}{ext}"
    return f"{prefix}_{first:%Y-%m-%d}_to_{last:%Y-%m-%d}{ext}"
 
 
def to_td(s):
    """'HH:MM:SS' text, Excel times, '0 days HH:MM:SS' or whole seconds -> timedelta. Blanks -> NaT."""
    if pd.api.types.is_timedelta64_dtype(s):
        return s
    txt = s.astype("string").str.strip()
    is_secs = txt.str.fullmatch(r"\d+").fillna(False).astype(bool)
    out = pd.to_timedelta(txt.mask(is_secs), errors="coerce")
    if is_secs.any():
        out[is_secs] = pd.to_timedelta(txt[is_secs].astype("int64"), unit="s")
    return out
 
 
def whole_seconds(td):
    """Timedelta -> whole seconds (cut, not rounded, like int(td.total_seconds())). NaT -> 0."""
    return np.floor(td.dt.total_seconds().fillna(0)).astype("int64")
 
 
def hms(seconds):
    """Whole seconds -> 'HH:MM:SS'."""
    s = pd.Series(seconds).fillna(0).astype("int64")
    h, m, sec = s // 3600, (s % 3600) // 60, s % 60
    return h.astype(str).str.zfill(2) + ":" + m.astype(str).str.zfill(2) + ":" + sec.astype(str).str.zfill(2)
 
 
def strip_suffixes(s):
    """First clean-up of dialer names (calls only): drop '(...)', '@domain' and ';...'."""
    return s.astype("string").str.replace(r"\s*\([^)]*\)|@.*|;.*", "", regex=True)
 
 
def normalise_name(s):
    """Second clean-up (calls and team list): single spaces, no '@...' or '(...', lower case."""
    return (s.astype("string")
             .str.replace(r"\s+", " ", regex=True).str.strip()
             .str.replace(r"@.*", "", regex=True)
             .str.replace(r"\(.*", "", regex=True)
             .str.strip().str.lower())
 
 
def is_connected(s):
    return s.astype("string").str.strip().str.lower().isin(["answered", "true"]).fillna(False)
def parse_stringee_time(s):
    """Real Excel date-times pass through; text like '10/28/2026 11:33:53 PM' is read month-first."""
    if pd.api.types.is_datetime64_any_dtype(s):
        return s
    txt = s.astype("string").str.strip()
    us = pd.to_datetime(txt, format="%m/%d/%Y %I:%M:%S %p", errors="coerce")
    iso = pd.to_datetime(txt, format="ISO8601", errors="coerce")
    return us.fillna(iso)

## **1. Stringee**

In [380]:
st_files = sorted(f for f in STRINGEE_DIR.glob("*.xlsx") if not f.name.startswith("~$"))
if not st_files:
    raise FileNotFoundError(f"No .xlsx files in {STRINGEE_DIR}")
print("Stringee files:", [f.name for f in st_files])

st_text = {"Account": str, "Customer number": str, "End of Call code": str, "Call status": str}
st_parts = [read_xlsx(f, dtype=st_text) for f in st_files]
st_raw = pd.concat(st_parts, ignore_index=True)
st_raw.columns = st_raw.columns.str.strip()

# 10/28/2026 09:34:25 PM  ->  month/day/year, 12-hour clock with AM/PM
st_raw["Start time"] = pd.to_datetime(st_raw["Start time"], format="%m/%d/%Y %I:%M:%S %p", errors="coerce")

unread = st_raw["Start time"].isna().sum()
if unread:
    raise ValueError(f"{unread} Stringee start times could not be read")

Stringee files: ['20261030_122403 (1).xlsx', '20261030_122526 (1).xlsx']


In [381]:
st_raw.sort_values('Start time', ascending = False)
st_raw['Start time'].min()

Timestamp('2026-10-01 00:06:24.268000')

In [382]:
if SAVE_PROCESSED_RAW:
    out_dir = PROCESSED_DIR / "st_output"
    out_dir.mkdir(parents=True, exist_ok=True)
    write_xlsx(st_raw, out_dir / date_range_name("st_daily", st_raw["Start time"], ".xlsx"))

In [383]:
st = st_raw.copy()
st["Account"] = st["Account"].str.strip()
st["Customer number"] = st["Customer number"].str.strip()
st["Start time"] = st["Start time"].dt.floor("s")
if EXCLUDE_STRINGEE_END_CODES:
    st = st[~st["End of Call code"].str.strip().isin(EXCLUDE_STRINGEE_END_CODES).fillna(False)]
st = st.drop_duplicates(subset=["Account", "Customer number", "Start time"], keep="first")

In [384]:
print("Stringee files")
for f, p in zip(st_files, st_parts):
    t = parse_stringee_time(p["Start time"])
    print(f"  {f.name}: {len(p):,} rows | {t.min()} -> {t.max()}")

print("Uno files")
for f in uno_files:
    d = pd.to_datetime(pd.read_csv(f, usecols=["Call Date"])["Call Date"], format="%d/%b/%Y")
    print(f"  {f.name}: {len(d):,} rows | {d.min():%Y-%m-%d} -> {d.max():%Y-%m-%d}")

print("\nCombined")
print(f"  Stringee: {st_raw['Start time'].min()} -> {st_raw['Start time'].max()}")
print(f"  Uno     : {uno['Call Date'].min():%Y-%m-%d} -> {uno['Call Date'].max():%Y-%m-%d}")

Stringee files
  20261030_122403 (1).xlsx: 176,295 rows | 2026-10-01 05:09:40.261000 -> 2026-10-28 23:33:53.172000
  20261030_122526 (1).xlsx: 234,976 rows | 2026-10-01 00:06:24.268000 -> 2026-10-28 23:06:13.965000
Uno files
  6ae2dc812e090a0fa580d6c9.csv: 2,064 rows | 2026-10-28 -> 2026-10-28

Combined
  Stringee: 2026-10-01 00:06:24.268000 -> 2026-10-28 23:33:53.172000
  Uno     : 2026-10-28 -> 2026-10-28


In [387]:
st_talk = to_td(st["Answer duration"]).fillna(pd.Timedelta(0))
st_calls = pd.DataFrame({
    "Source": "Stringee",
    "Date": st["Start time"].dt.normalize(),
    "Dialer Name": st["Account"],
    "Number": st["Customer number"],
    "Call Status": np.where(is_connected(st["Call status"]), "connected", "not connected"),
    "Call Start Time": st["Start time"],
    "Total Call Duration": to_td(st["Queue duration"]).fillna(pd.Timedelta(0)) + st_talk,
    "Talk Time": st_talk,
    "Hold Time": to_td(st["Hold duration"]),
})
print("Stringee rows:", " + ".join(f"{len(p):,}" for p in st_parts), "=", f"{len(st_raw):,}",
      "->", f"{len(st_calls):,} after end-code filter and de-duplication")

Stringee rows: 176,295 + 234,976 = 411,271 -> 411,108 after end-code filter and de-duplication


## **2. Uno**

In [388]:
uno_files = sorted(UNO_DIR.glob("*.csv"))
if not uno_files:
    raise FileNotFoundError(f"No .csv files in {UNO_DIR}")
print("Uno files:", [f.name for f in uno_files])

uno = pd.concat([pd.read_csv(f, dtype={"Caller Number": str, "Caller Ids": str, "Contact Info": str},
                             low_memory=False) for f in uno_files], ignore_index=True)
if "Call Id" in uno.columns:                     # overlapping exports: keep each call once
    uno = uno.drop_duplicates(subset="Call Id")
uno["Call Date"] = pd.to_datetime(uno["Call Date"], format="%d/%b/%Y")

Uno files: ['6ae2dc812e090a0fa580d6c9.csv']


In [389]:
if SAVE_PROCESSED_RAW:
    out_dir = PROCESSED_DIR / "uno_output"
    out_dir.mkdir(parents=True, exist_ok=True)
    write_xlsx(uno, out_dir / date_range_name("uno", uno["Call Date"], ".xlsx"))

In [390]:
uno_start = pd.to_datetime(
    uno["Call Date"].dt.strftime("%Y-%m-%d") + " " + uno["Call Time"].astype(str).str.strip().str.upper(),
    format="%Y-%m-%d %I:%M:%S %p", errors="coerce")
 
if UNO_CONNECTED_RULE == "both_answered":
    uno_connected = uno["Status"].astype("string").str.strip().str.lower().eq("both answered").fillna(False)
else:
    uno_connected = is_connected(uno["Answered"])

In [391]:
uno_calls = pd.DataFrame({
    "Source": "Uno",
    "Date": uno["Call Date"],
    "Dialer Name": uno["Contact Info"].str.extract(r"\bagent_id\s*:\s*(\d+)", expand=False),
    "Number": uno["Contact Info"].str.extract(r"customer\s*:\s*(\d+)", expand=False),
    "Call Status": np.where(uno_connected, "connected", "not connected"),
    "Call Start Time": uno_start,
    "Total Call Duration": to_td(uno["Call Duration"]),
    "Talk Time": to_td(uno["Talk Duration"]),
    "Hold Time": to_td(uno["Hold Duration"]),
})
print(f"Uno rows: {len(uno_calls):,} | start times that failed to parse: {uno_start.isna().sum()}")

Uno rows: 2,064 | start times that failed to parse: 0


In [392]:
if {"Status", "Answered"} <= set(uno.columns):
    print("\nUno Status vs Answered (check which calls really connected):")
    print(pd.crosstab(uno["Status"], uno["Answered"], margins=True))


Uno Status vs Answered (check which calls really connected):
Answered                     False  True   All
Status                                        
Agent Failed                     6     0     6
Agent Unanswered                34     0    34
Both Answered                    0   432   432
Cust. Ans                       14     0    14
Cust. Ans - Agent Failed        12     0    12
Cust. Ans - Agent Not Found      1     0     1
Cust. Ans. - Agent Unans.       12     0    12
Cust. Failed - Agent Ans.        0   124   124
Cust. Hangup                     3     0     3
Cust. Unans - Agent Ans.         0   129   129
From Ans. - To Failed            0   398   398
From Ans. - To Unans.            0   601   601
From Busy                       75     0    75
From Failed                      1     0     1
From Unanswered                217     0   217
All                            375  1684  2059


## **3. Combine and match to the team list**

In [393]:
calls = pd.concat([uno_calls, st_calls], ignore_index=True)
calls["Hold Time"] = calls["Hold Time"].fillna(pd.Timedelta(0))
calls["Dialer Name"] = strip_suffixes(calls["Dialer Name"])
calls = calls[calls["Dialer Name"].notna() & calls["Dialer Name"].ne("---")].copy()
calls["Dialer Name"] = normalise_name(calls["Dialer Name"])
 
ref = pd.read_csv(TEAM_LIST, dtype={"Dialer Name": str})
ref = ref[~ref["Email"].str.contains("inactive", case=False, na=False)].copy()
ref["Dialer Name"] = normalise_name(ref["Dialer Name"])
ref = ref.drop_duplicates(subset=["Dialer Name", "Email"]).rename(columns={"Email": "CRM ID"})
 
shared = ref[ref["Dialer Name"].notna() & ref.duplicated("Dialer Name", keep=False)]
if len(shared):
    print("WARNING: these dialer names belong to more than one CRM ID, so their calls are counted for each:")
    print(shared[["Dialer Name", "CRM ID"]].sort_values("Dialer Name").to_string(index=False))
 
merged = calls.merge(ref, how="left", on="Dialer Name")
not_found = merged[merged["CRM ID"].isna()].copy()
print(f"Calls matched: {merged['CRM ID'].notna().sum():,} | not found in team list: {len(not_found):,}")

Dialer Name                CRM ID
      aamir     aamir@nexora.live
      aamir                 aamir
  chandan.y chandan.y@nexora.live
  chandan.y             chandan.y
    divya.d   divya.d@nexora.live
    divya.d               divya.d
      irfan     irfan@nexora.live
      irfan                 irfan
      mahek     mahek@nexora.live
      mahek                 mahek
       nair      nair@nexora.live
       nair                  nair
      rajan     rajan@nexora.live
      rajan                 rajan
      ramya                 ramya
      ramya     ramya@nexora.live
     riddhi    riddhi@nexora.live
     riddhi                riddhi
      rudra     rudra@nexora.live
      rudra                 rudra
  sangita.m sangita.m@nexora.live
  sangita.m             sangita.m
  saurabh.s saurabh.s@nexora.live
  saurabh.s             saurabh.s
     shruti    shruti@nexora.live
     shruti                shruti
      zaina     zaina@nexora.live
      zaina                 zaina
Calls matched:

## **4. Call gaps**

In [394]:
dialers = merged[merged["CRM ID"].notna() & merged["Talk Time"].notna()].copy()
if DEDUPE_NUMBER_AND_TIME:
    dialers = dialers.drop_duplicates(subset=["Number", "Call Start Time"])
dialers["Total Call Duration"] = dialers["Total Call Duration"].fillna(pd.Timedelta(0))
dialers = dialers.sort_values(["Date", "CRM ID", "Call Start Time"], kind="mergesort").reset_index(drop=True)
 
by_cre_day = dialers.groupby(["Date", "CRM ID"], sort=False)
prev_start = by_cre_day["Call Start Time"].shift()
prev_end = prev_start + by_cre_day["Total Call Duration"].shift()
gap = (dialers["Call Start Time"] - prev_end).clip(lower=pd.Timedelta(0))
gap_seconds = whole_seconds(gap)
 
start_tod = dialers["Call Start Time"] - dialers["Call Start Time"].dt.normalize()
prev_tod = prev_start - prev_start.dt.normalize()
in_hours = start_tod.between(WORK_START, WORK_END) & prev_tod.between(WORK_START, WORK_END)
flagged = in_hours & (gap.dt.total_seconds() > GAP_FLAG_SECONDS).fillna(False)
 
dialers["Call Gap"] = np.where(flagged, "Yes", "No")
dialers["Gap Duration"] = hms(gap_seconds)
dialers["hour"] = dialers["Call Start Time"].dt.hour
slot = (dialers["hour"].fillna(99) - 7).clip(0, len(SLOT_LABELS) - 1).astype(int)   # unknown time -> '20+', as before
dialers["Interval"] = np.array(SLOT_LABELS)[slot]

In [395]:
talk_s = whole_seconds(dialers["Talk Time"])
hold_s = whole_seconds(dialers["Hold Time"])
conn = dialers["Call Status"].eq("connected")
long_conn = conn & (talk_s > LONG_CALL_SECONDS)
calc = pd.DataFrame({
    "CRM ID": dialers["CRM ID"], "Date": dialers["Date"],
    "Call Status": dialers["Call Status"], "Number": dialers["Number"],
    "connected": conn.astype(int), "gap_flag": flagged.astype(int), "long_conn": long_conn.astype(int),
    "total_s": whole_seconds(dialers["Total Call Duration"]),
    "talk_conn_s": talk_s.where(conn, 0), "talk_long_s": talk_s.where(long_conn, 0),
    "hold_conn_s": hold_s.where(conn, 0), "gap_s": gap_seconds,
})
print(f"Calls in the report: {len(dialers):,} | unknown start times: {dialers['Call Start Time'].isna().sum()}")

Calls in the report: 366,954 | unknown start times: 0


## **5. Summary: one row per CRE per day**

In [396]:
A = calc.groupby(["CRM ID", "Date"]).agg(
    Total_Dialed_Calls=("Call Status", "count"),
    Unique_Dialed_Numbers=("Number", "nunique"),
    Total_Connected_Calls=("connected", "sum"),
    Total_Number_of_Call_Gap=("gap_flag", "sum"),
    Total_Call_GT_30=("long_conn", "sum"),
    Total_Duration=("total_s", "sum"),
    Total_Talk_Time=("talk_conn_s", "sum"),
    Total_Talk_Time_GT_30=("talk_long_s", "sum"),
    Total_Connected_Hold_Time=("hold_conn_s", "sum"),
    Total_Gap_Duration=("gap_s", "sum"),
).reset_index()
 
A["Total_Gap_Duration"] = np.where(A["Total_Gap_Duration"] > BREAK_ALLOWANCE,
                                   A["Total_Gap_Duration"] - BREAK_ALLOWANCE, A["Total_Gap_Duration"])
A["Avg_Gap_per_call"] = A["Total_Gap_Duration"] / A["Total_Dialed_Calls"]
A["Gap Duration After Leverage"] = (A["Total_Gap_Duration"] - A["Total_Dialed_Calls"] * LEVERAGE_PER_CALL).clip(lower=0)
login_s = A["Total_Duration"] + A["Total_Gap_Duration"] + BREAK_ALLOWANCE
A["Login Hours"] = hms(login_s)
login_td = pd.to_timedelta(login_s, unit="s")
A["Attendance"] = np.select([login_td < limit for limit, _ in ATTENDANCE_BANDS],
                            [label for _, label in ATTENDANCE_BANDS], default="Present")

In [397]:
crm_ref = ref.drop_duplicates(subset=["CRM ID"])[["CRM ID", "Employee code", "Full Name", "Pool", "TL", "Vertical"]]
grid = pd.MultiIndex.from_product([crm_ref["CRM ID"], A["Date"].unique()], names=["CRM ID", "Date"]).to_frame(index=False)
summary = crm_ref.merge(grid.merge(A, how="left", on=["CRM ID", "Date"]).fillna(0), how="outer", on="CRM ID")
 
count_cols = ["Total_Dialed_Calls", "Unique_Dialed_Numbers", "Total_Connected_Calls", "Total_Number_of_Call_Gap",
              "Total_Call_GT_30", "Total_Duration", "Total_Talk_Time", "Total_Talk_Time_GT_30",
              "Total_Connected_Hold_Time", "Total_Gap_Duration"]
summary[count_cols] = summary[count_cols].fillna(0).astype(int)
lead = ["Date", "Pool", "TL", "CRM ID", "Employee code", "Full Name", "Vertical"]
summary = summary[lead + [c for c in summary.columns if c not in lead]]
summary = summary.dropna(subset=["CRM ID", "Date", "Avg_Gap_per_call", "Gap Duration After Leverage"])

In [398]:
for col in ["Total_Duration", "Total_Talk_Time", "Total_Talk_Time_GT_30", "Total_Connected_Hold_Time",
            "Total_Gap_Duration", "Avg_Gap_per_call", "Gap Duration After Leverage"]:
    summary[col] = hms(np.round(summary[col].astype(float)))
summary["Login Hours"] = summary["Login Hours"].replace(0, "00:00:00")
summary["Attendance"] = summary["Attendance"].replace(0, "Absent")
if not KEEP_ATTENDANCE:
    summary = summary.drop(columns="Attendance")
summary = summary.sort_values(["Date", "TL", "CRM ID"], kind="mergesort").reset_index(drop=True)
summary

,Date,Pool,TL,CRM ID,Employee code,Full Name,Vertical,Total_Dialed_Calls,Unique_Dialed_Numbers,Total_Connected_Calls,Total_Number_of_Call_Gap,Total_Call_GT_30,Total_Duration,Total_Talk_Time,Total_Talk_Time_GT_30,Total_Connected_Hold_Time,Total_Gap_Duration,Avg_Gap_per_call,Gap Duration After Leverage,Login Hours
0,2026-10-01,Sales,Ashutosh,ahana.a@nexora.live,E-2297,ahana.a,TradeX,322,299,98,57,36,03:38:13,01:04:29,00:51:02,00:00:00,05:15:58,00:00:59,01:14:28,09:54:11
1,2026-10-01,Sales,Ashutosh,apurva@nexora.live,E-2405,apurva,TradeX,0,0,0,0,0,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00
2,2026-10-01,Sales,Ashutosh,ashu,E-2132,ashu,TradeX,261,203,90,64,43,04:12:17,02:01:39,01:50:17,00:00:00,04:00:52,00:00:55,00:45:07,09:13:09
3,2026-10-01,Sales,Ashutosh,harshita,NaN,harshita,TradeX,141,113,34,48,16,01:44:59,00:38:15,00:34:06,00:00:00,02:20:17,00:01:00,00:34:32,05:05:16
4,2026-10-01,Sales,Ashutosh,kavya,NaN,kavya,TradeX,304,292,94,54,32,03:54:11,01:33:18,01:19:00,00:00:00,04:46:18,00:00:57,00:58:18,09:40:29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3299,2026-10-28,Pull back-WFH,Swathi,rajan@nexora.live,E-2243,Suvam Gandhi,TradeX,150,120,27,63,15,02:13:26,01:12:14,01:08:48,00:00:00,04:57:01,00:01:59,03:04:31,08:10:27
3300,2026-10-28,Pull back-WFH,Swathi,rudra@nexora.live,E-2365,rudra,TradeX,206,184,23,76,8,01:37:05,00:13:03,00:08:25,00:00:00,05:46:53,00:01:41,03:12:23,08:23:58
3301,2026-10-28,Pull back-WFH,Swathi,sikander@nexora.live,E-1785,Sagar Prajwal D'souza,TradeX,22,21,4,2,1,00:15:32,00:01:59,00:01:34,00:00:00,00:10:15,00:00:28,00:00:00,01:25:47
3302,2026-10-28,Pull back-WFH,Swathi,zaid.k@nexora.live,E-1275,Zaid,TradeX,295,286,77,65,35,03:15:27,01:15:33,01:06:08,00:00:00,04:56:43,00:01:00,01:15:28,09:12:10


## **6. Hourly report per CRE**

In [399]:
keys = ["Date", "CRM ID", "Employee code", "Full Name", "Pool", "TL"]
pieces = {
    "Calls": dialers.pivot_table(index=keys, columns="Interval", values="Call Status", aggfunc="count", fill_value=0),
    "Unique Calls": dialers.pivot_table(index=keys, columns="Interval", values="Number", aggfunc="nunique", fill_value=0),
    "Talk Time": dialers.pivot_table(index=keys, columns="Interval", values="Talk Time", aggfunc="sum",
                                     fill_value=pd.Timedelta(0)),
    "Total Call Duration": dialers.pivot_table(index=keys, columns="Interval", values="Total Call Duration",
                                               aggfunc="sum", fill_value=pd.Timedelta(0)),
}
for metric, table in pieces.items():
    table.columns = [f"{slot} - {metric}" for slot in table.columns]
hourly = pd.concat(pieces.values(), axis=1).reset_index()
 
slots = [s for s in SLOT_LABELS if f"{s} - Calls" in hourly.columns]
metrics = ["Calls", "Unique Calls", "Talk Time", "Total Call Duration"]
hourly = hourly[keys + [f"{s} - {m}" for s in slots for m in metrics if f"{s} - {m}" in hourly.columns]]
 
hourly["Total Calls"] = hourly[[f"{s} - Calls" for s in slots]].sum(axis=1)
hourly["Total Unique Calls"] = hourly[[f"{s} - Unique Calls" for s in slots]].sum(axis=1)
hourly["Total Talk Time"] = hourly[[f"{s} - Talk Time" for s in slots]].sum(axis=1)
hourly["Total Call Duration"] = hourly[[f"{s} - Total Call Duration" for s in slots]].sum(axis=1)

In [400]:
pool1 = hourly["Pool"] == 1
for s in slots:
    calls_n = hourly[f"{s} - Calls"]
    secs = hourly[f"{s} - Talk Time"].dt.total_seconds()
    mins = secs // 3600 * 60 + (secs % 3600) // 60 + (secs % 60) / 60
    target = (calls_n >= SLOT_TARGET) | (mins >= SLOT_TARGET)
    warning = np.where(pool1, calls_n.between(*POOL1_WARNING) | mins.between(*POOL1_WARNING),
                       calls_n.between(*OTHER_WARNING) | mins.between(*OTHER_WARNING))
    hourly[f"{s} - Status"] = np.select([target, warning], ["Complete", "Warning"], default="Not Complete")
 
for col in [c for c in hourly.columns if "Talk Time" in c or "Total Call Duration" in c]:
    hourly[col] = hms(whole_seconds(hourly[col]))
 
first_last = (dialers.groupby(["CRM ID", "Date"])["Call Start Time"].agg(["min", "max"])
                     .rename(columns={"min": "First Call Time", "max": "Last Call Time"}).reset_index())
on = ["CRM ID", "Date"]
hourly = (hourly.merge(summary[on + ["Login Hours"]], how="left", on=on)
                .merge(first_last, how="left", on=on)
                .merge(summary[on + ["Total_Connected_Calls"]], how="left", on=on)
                .sort_values(["Date", "TL", "CRM ID"], kind="mergesort").reset_index(drop=True))

check = hourly[on + ["Total Calls"]].merge(summary[on + ["Total_Dialed_Calls"]], on=on, how="left")
mismatch = check[check["Total Calls"] != check["Total_Dialed_Calls"]]
print("Hourly totals match the summary:", mismatch.empty)
if not mismatch.empty:
    print(mismatch.to_string(index=False))
hourly

Hourly totals match the summary: True


,Date,CRM ID,Employee code,Full Name,Pool,TL,8 to 9 AM - Calls,8 to 9 AM - Unique Calls,8 to 9 AM - Talk Time,8 to 9 AM - Total Call Duration,...,15 to 16 PM - Status,16 to 17 PM - Status,17 to 18 PM - Status,18 to 19 PM - Status,19 to 20 PM - Status,20+ - Status,Login Hours,First Call Time,Last Call Time,Total_Connected_Calls
0,2026-10-01,ahana.a@nexora.live,E-2297,ahana.a,Sales,Ashutosh,0,0,00:00:00,00:00:00,...,Not Complete,Complete,Complete,Complete,Not Complete,Not Complete,09:54:11,2026-10-01 09:36:21,2026-10-01 19:29:45,98
1,2026-10-01,ashu,E-2132,ashu,Sales,Ashutosh,0,0,00:00:00,00:00:00,...,Complete,Complete,Complete,Warning,Not Complete,Not Complete,09:13:09,2026-10-01 09:51:14,2026-10-01 19:04:07,90
2,2026-10-01,rakshit@nexora.live,E-2404,rakshit,Sales,Ashutosh,0,0,00:00:00,00:00:00,...,Not Complete,Not Complete,Not Complete,Not Complete,Not Complete,Not Complete,02:53:57,2026-10-01 10:01:55,2026-10-01 12:55:03,12
3,2026-10-01,vamshi@nexora.live,E-2297,vamshi,Sales,Ashutosh,0,0,00:00:00,00:00:00,...,Not Complete,Complete,Complete,Not Complete,Not Complete,Not Complete,09:52:38,2026-10-01 09:42:35,2026-10-01 19:31:43,111
4,2026-10-01,zara,E-2362,zara,Sales,Ashutosh,0,0,00:00:00,00:00:00,...,Not Complete,Complete,Complete,Complete,Not Complete,Not Complete,10:01:40,2026-10-01 09:33:21,2026-10-01 19:34:11,101
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1483,2026-10-28,rajan@nexora.live,E-2243,Suvam Gandhi,Pull back-WFH,Swathi,0,0,00:00:00,00:00:00,...,Not Complete,Not Complete,Not Complete,Not Complete,Not Complete,Not Complete,08:10:27,2026-10-28 09:39:40,2026-10-28 17:50:06,27
1484,2026-10-28,rudra@nexora.live,E-2365,rudra,Pull back-WFH,Swathi,0,0,00:00:00,00:00:00,...,Not Complete,Not Complete,Complete,Not Complete,Not Complete,Not Complete,08:23:58,2026-10-28 09:30:05,2026-10-28 17:54:03,23
1485,2026-10-28,sikander@nexora.live,E-1785,Sagar Prajwal D'souza,Pull back-WFH,Swathi,0,0,00:00:00,00:00:00,...,Not Complete,Not Complete,Not Complete,Not Complete,Not Complete,Not Complete,01:25:47,2026-10-28 11:23:08,2026-10-28 11:48:35,4
1486,2026-10-28,zaid.k@nexora.live,E-1275,Zaid,Pull back-WFH,Swathi,0,0,00:00:00,00:00:00,...,Complete,Not Complete,Complete,Not Complete,Not Complete,Not Complete,09:12:10,2026-10-28 09:30:47,2026-10-28 18:42:36,77


## **7. Call-level detail and not-found list**

In [401]:

 
dialer_out = dialers.drop(columns=["Dialer"], errors="ignore")
for col in ["Total Call Duration", "Talk Time", "Hold Time"]:
    dialer_out[col] = hms(whole_seconds(dialer_out[col]))
dialer_out["Number"] = ("'" + dialer_out["Number"].astype("string")).str.split(".").str[0]
dialer_out["Connected"] = np.where(dialer_out["Call Status"].eq("connected"), "1", "")
dialer_out["Not Connected"] = np.where(dialer_out["Call Status"].eq("not connected"), "1", "")
 
for col in ["Total Call Duration", "Talk Time", "Hold Time"]:
    not_found[col] = hms(whole_seconds(not_found[col]))
 
dates_by_source = dialers.groupby("Source")["Date"].agg(lambda d: ", ".join(sorted(d.dt.strftime("%Y-%m-%d").unique())))
print("Dates per source:", dates_by_source.to_dict())
print("Calls per day:", dialers.groupby(dialers["Date"].dt.strftime("%Y-%m-%d")).size().to_dict())
print("\nNames not found in the team list (most calls first):")
print(not_found.groupby(["Source", "Dialer Name"]).size().sort_values(ascending=False).head(20))

Dates per source: {'Stringee': '2026-10-01, 2026-10-02, 2026-10-03, 2026-10-04, 2026-10-05, 2026-10-06, 2026-10-07, 2026-10-08, 2026-10-09, 2026-10-10, 2026-10-11, 2026-10-12, 2026-10-13, 2026-10-14, 2026-10-15, 2026-10-16, 2026-10-17, 2026-10-18, 2026-10-19, 2026-10-20, 2026-10-21, 2026-10-22, 2026-10-23, 2026-10-24, 2026-10-25, 2026-10-26, 2026-10-27, 2026-10-28'}
Calls per day: {'2026-10-01': 14294, '2026-10-02': 6932, '2026-10-03': 6098, '2026-10-04': 2, '2026-10-05': 16154, '2026-10-06': 16605, '2026-10-07': 17815, '2026-10-08': 16272, '2026-10-09': 17143, '2026-10-10': 10347, '2026-10-11': 3511, '2026-10-12': 17955, '2026-10-13': 15927, '2026-10-14': 17194, '2026-10-15': 15426, '2026-10-16': 13658, '2026-10-17': 8216, '2026-10-18': 4612, '2026-10-19': 15950, '2026-10-20': 5918, '2026-10-21': 17659, '2026-10-22': 17348, '2026-10-23': 17304, '2026-10-24': 10348, '2026-10-25': 5012, '2026-10-26': 20090, '2026-10-27': 20196, '2026-10-28': 18968}

Names not found in the team list (mos

In [402]:
def days(s):
    return s.dt.strftime("%Y-%m-%d").value_counts().sort_index().to_dict()

print("Stringee, per file:")
for f, p in zip(st_files, st_parts):
    print("  ", f.name, days(pd.to_datetime(p["Start time"], format="ISO8601")))

print("Uno, per file:")
for f in uno_files:
    u = pd.read_csv(f, usecols=["Call Date"])
    print("  ", f.name, days(pd.to_datetime(u["Call Date"], format="%d/%b/%Y")))

print("Stringee after filters:", days(st_calls["Date"]))
print("Uno calls             :", days(uno_calls["Date"]))
print("Matched to team list  :", days(merged.loc[merged["CRM ID"].notna(), "Date"]))
print("In the report         :", days(dialers["Date"]))
print("Summary               :", days(summary["Date"]))

Stringee, per file:
   20261030_122403 (1).xlsx {'2026-10-01': 7114, '2026-10-02': 4600, '2026-10-03': 2675, '2026-10-04': 43, '2026-10-05': 7490, '2026-10-06': 7508, '2026-10-07': 8103, '2026-10-08': 7517, '2026-10-09': 8073, '2026-10-10': 5322, '2026-10-11': 2006, '2026-10-12': 8597, '2026-10-13': 7245, '2026-10-14': 9024, '2026-10-15': 7896, '2026-10-16': 7123, '2026-10-17': 4318, '2026-10-18': 2202, '2026-10-19': 7155, '2026-10-20': 2668, '2026-10-21': 7842, '2026-10-22': 8448, '2026-10-23': 8162, '2026-10-24': 5398, '2026-10-25': 2766, '2026-10-26': 9483, '2026-10-27': 8798, '2026-10-28': 8719}
   20261030_122526 (1).xlsx {'2026-10-01': 8727, '2026-10-02': 2988, '2026-10-03': 4096, '2026-10-04': 33, '2026-10-05': 10128, '2026-10-06': 10592, '2026-10-07': 11486, '2026-10-08': 10529, '2026-10-09': 10597, '2026-10-10': 6251, '2026-10-11': 2016, '2026-10-12': 11002, '2026-10-13': 10638, '2026-10-14': 10511, '2026-10-15': 9831, '2026-10-16': 8523, '2026-10-17': 5218, '2026-10-18': 3322

## **8. Export**

In [403]:
REPORT_DIR.mkdir(parents=True, exist_ok=True)

reports = {
    "Summary": summary,
    "Dialer": dialer_out,
    "Not_Found_Users": not_found,
    "CRE_dial_coun": hourly,
}
for prefix, df in reports.items():
    name = date_range_name(prefix, dialers["Date"], ".csv")
    df.to_csv(REPORT_DIR / name, index=False)
    print("Saved:", name)

Saved: Summary_2026-10-01_to_2026-10-28.csv
Saved: Dialer_2026-10-01_to_2026-10-28.csv
Saved: Not_Found_Users_2026-10-01_to_2026-10-28.csv
Saved: CRE_dial_coun_2026-10-01_to_2026-10-28.csv
